## Inicio de una sesión PySpark

Para empezar, necesitaremos crear una sesión PySpark. No entraremos en mucho detalle aquí porque ya lo explicamos en la sesión anterior.

In [1]:
import pyspark
import os
from pyspark.sql import SparkSession

# Creamos una sesión PySpark contra nuestro servicio "spark-master"
spark = SparkSession.builder \
    .master(os.environ.get('SPARK_MASTER')) \
    .appName("pyspark-test") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 13:14:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Descarga de datos

A continuación, descargamos el fichero de "gran volumen de datos" de taxis FHV (For-Hire Vehicles) desde la web de [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

In [2]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-01.parquet

File ‘fhvhv_tripdata_2026-01.parquet’ already there; not retrieving.



## Lectura de datos

Ahora, vamos a leer los datos descargados. Como en este caso los datos los estamos descargando en formato Parquet, el código va a ser ligeramente diferente a lo que hicimos en la sesión anterior.

In [3]:
# Leemos el fichero Parquet
df = spark.read.parquet('fhvhv_tripdata_2026-01.parquet')

# Mostramos los primeros 3 registros
df.head(3)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B03404', originating_base_num='B03404', request_datetime=datetime.datetime(2026, 1, 1, 0, 50, 37), on_scene_datetime=datetime.datetime(2026, 1, 1, 0, 52, 31), pickup_datetime=datetime.datetime(2026, 1, 1, 0, 54, 30), dropoff_datetime=datetime.datetime(2026, 1, 1, 1, 13, 23), PULocationID=262, DOLocationID=79, trip_miles=4.3, trip_time=1133, base_passenger_fare=31.24, tolls=0.0, bcf=0.75, sales_tax=2.77, congestion_surcharge=2.75, airport_fee=0.0, tips=0.0, driver_pay=21.1, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag='N', wav_request_flag='N', wav_match_flag='N', cbd_congestion_fee=1.5),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B03404', originating_base_num='B03404', request_datetime=datetime.datetime(2026, 1, 1, 0, 9, 12), on_scene_datetime=datetime.datetime(2026, 1, 1, 0, 12, 17), pickup_datetime=datetime.datetime(2026, 1, 1, 0, 12, 39), dropoff_datetime=datetime.datetime(2026, 1, 1, 0, 22, 42)

### Tipos de datos

Gracias a que los datos de los ficheros Parquet vienen tipados, las columnas con datos temporales tienen ya parseados como fechas. Para comprobarlo, podemos usar la propiedad `schema` de los _dataframes_ de PySpark.

In [4]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('sha

### Proceso de ficheros en paralelo

En un proceso típico de Spark, tendríamos una cola de trabajos, por ejemplo ficheros a procesar, y la distribuiríamos de forma lo más equitativamente posible a través de los nodos de nuestro clúster Spark.

Sin embargo, en nuestro proceso actual tenemos un único fichero, de forma que no tenemos trabajor que distribuir. Para resolver esto, dividiremos el fichero en partes pequeñas. A estas partes, en Spark, las llamamos **particiones**. Para dividir nuestro _dataframe_ en partes, podemos usar `repartition`.

In [5]:
df.repartition(8)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp_ntz, on_scene_datetime: timestamp_ntz, pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int, trip_miles: double, trip_time: bigint, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string, cbd_congestion_fee: double]

Ahora, para probar la paralelización, podemos simplemente guardar el _dataframe_ en una carpeta.

In [9]:
df.write.mode("overwrite").parquet('fhv/2026/01')

In [7]:
!ls -lah fhv/2026/01

total 556M
drwxr-xr-x 2 root root 4.0K Mar  1 13:09 .
drwxr-xr-x 3 root root 4.0K Mar  1 13:08 ..
-rw-r--r-- 1 root root  27M Mar  1 13:09 part-00000-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet
-rw-r--r-- 1 root root 212K Mar  1 13:09 .part-00000-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  27M Mar  1 13:09 part-00001-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet
-rw-r--r-- 1 root root 210K Mar  1 13:09 .part-00001-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  62M Mar  1 13:09 part-00002-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet
-rw-r--r-- 1 root root 490K Mar  1 13:09 .part-00002-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet.crc
-rw-r--r-- 1 root root  28M Mar  1 13:09 part-00003-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet
-rw-r--r-- 1 root root 221K Mar  1 13:09 .part-00003-0a3b3907-00c8-405c-a634-901f86c06114-c000.snappy.parquet.crc
-rw-r--r--